# Human Resources Logic

This notebook builds AI logic for the Human Resources module: staff roster generation, role capacity, fatigue monitoring, shift compliance, auto-assignment to flights, shortage detection, crew activity feed, and HR recommendations.

## 1. Setup

In [ ]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

now = datetime(2026, 4, 27, 9, 0)

## 2. HR Configuration

In [ ]:
ROLES = [
    'Ramp Agents', 'Baggage Handlers', 'Cleaning Crew', 'Fuelers',
    'Caterers', 'Boarding Pass Crew', 'Technician', 'Pushback Drivers'
]

ROLE_CONFIG = {
    'Ramp Agents': {'base_count': 42, 'max_hours': 8, 'max_continuous_hours': 4, 'required_rest_hours': 8, 'fatigue_rate': 7.5, 'skills': ['marshalling', 'walkaround', 'chocks']},
    'Baggage Handlers': {'base_count': 54, 'max_hours': 8, 'max_continuous_hours': 4, 'required_rest_hours': 8, 'fatigue_rate': 8.2, 'skills': ['loading', 'unloading', 'belt_ops']},
    'Cleaning Crew': {'base_count': 38, 'max_hours': 8, 'max_continuous_hours': 3, 'required_rest_hours': 8, 'fatigue_rate': 9.1, 'skills': ['cabin_cleaning', 'lavatory', 'deep_clean']},
    'Fuelers': {'base_count': 20, 'max_hours': 8, 'max_continuous_hours': 5, 'required_rest_hours': 10, 'fatigue_rate': 6.8, 'skills': ['fueling', 'quality_check', 'safety']},
    'Caterers': {'base_count': 22, 'max_hours': 8, 'max_continuous_hours': 4, 'required_rest_hours': 8, 'fatigue_rate': 7.4, 'skills': ['meal_loading', 'inventory', 'cold_chain']},
    'Boarding Pass Crew': {'base_count': 30, 'max_hours': 8, 'max_continuous_hours': 5, 'required_rest_hours': 8, 'fatigue_rate': 6.5, 'skills': ['boarding', 'customer_service', 'special_assistance']},
    'Technician': {'base_count': 18, 'max_hours': 9, 'max_continuous_hours': 6, 'required_rest_hours': 12, 'fatigue_rate': 5.8, 'skills': ['inspection', 'avionics', 'mechanical']},
    'Pushback Drivers': {'base_count': 24, 'max_hours': 8, 'max_continuous_hours': 5, 'required_rest_hours': 8, 'fatigue_rate': 6.9, 'skills': ['pushback', 'towbar', 'radio']},
}

FIRST_NAMES = ['Arjun', 'Priya', 'Rahul', 'Sneha', 'Vikram', 'Anita', 'Raj', 'Pooja', 'Suresh', 'Divya', 'Karan', 'Meena', 'Anil', 'Nisha', 'Deepak']
LAST_NAMES = ['Sharma', 'Patel', 'Singh', 'Kumar', 'Gupta', 'Mehta', 'Joshi', 'Verma', 'Nair', 'Reddy']
SHIFT_WINDOWS = {
    'Morning': (datetime(2026, 4, 27, 6, 0), datetime(2026, 4, 27, 14, 0)),
    'Evening': (datetime(2026, 4, 27, 14, 0), datetime(2026, 4, 27, 22, 0)),
    'Night': (datetime(2026, 4, 27, 22, 0), datetime(2026, 4, 28, 6, 0)),
}

## 3. Generate Staff Roster

In [ ]:
def random_name():
    return f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"


def generate_staff_roster():
    rows = []
    employee_number = 1001

    for role, cfg in ROLE_CONFIG.items():
        for _ in range(cfg['base_count']):
            shift = random.choices(['Morning', 'Evening', 'Night'], weights=[45, 40, 15])[0]
            shift_start, shift_end = SHIFT_WINDOWS[shift]
            hours_worked = max(0, round((now - shift_start).total_seconds() / 3600, 1)) if shift_start <= now <= shift_end else random.uniform(0, cfg['max_hours'])
            hours_worked = round(min(hours_worked, cfg['max_hours'] + random.choice([0, 0, 0.5, 1.0])), 1)
            continuous_hours = round(min(hours_worked, random.uniform(1, cfg['max_continuous_hours'] + 1.5)), 1)
            base_fatigue = 18 + hours_worked * cfg['fatigue_rate'] + continuous_hours * 4
            fatigue = int(np.clip(np.random.normal(base_fatigue, 9), 5, 98))
            status = random.choices(['active', 'available', 'on_leave', 'sick', 'break'], weights=[50, 26, 8, 4, 12])[0]
            if fatigue >= 82 and status in ['active', 'available']:
                status = random.choice(['active', 'break'])

            rows.append({
                'employee_id': f'EMP-{employee_number}',
                'name': random_name(),
                'role': role,
                'skills': ','.join(cfg['skills']),
                'shift': shift,
                'shift_start': shift_start,
                'shift_end': shift_end,
                'status': status,
                'hours_worked': hours_worked,
                'continuous_hours': continuous_hours,
                'max_hours': cfg['max_hours'],
                'max_continuous_hours': cfg['max_continuous_hours'],
                'required_rest_hours': cfg['required_rest_hours'],
                'last_rest_hours': round(random.uniform(4, 14), 1),
                'fatigue_score': fatigue,
                'current_gate': random.choice([f'G{str(i).zfill(2)}' for i in range(1, 25)] + ['Standby', 'Crew Room']),
            })
            employee_number += 1

    return pd.DataFrame(rows)


staff_df = generate_staff_roster()
staff_df.head(10)

## 4. Generate Flight Crew Demand

In [ ]:
TASK_REQUIREMENTS = {
    'Ramp Agents': 3,
    'Baggage Handlers': 4,
    'Cleaning Crew': 3,
    'Fuelers': 1,
    'Caterers': 2,
    'Boarding Pass Crew': 2,
    'Technician': 1,
    'Pushback Drivers': 1,
}

AIRCRAFT_SIZE_FACTOR = {
    'Regional': 0.75,
    'Narrow Body': 1.0,
    'Wide Body': 1.8,
}


def generate_flight_tasks(n=34):
    flights = []
    for i in range(n):
        aircraft_size = random.choices(['Regional', 'Narrow Body', 'Wide Body'], weights=[18, 64, 18])[0]
        gate = f'G{random.randint(1, 24):02d}'
        arrival = now + timedelta(minutes=random.randint(-60, 180))
        departure = arrival + timedelta(minutes=random.randint(45, 115))
        priority = random.choices(['Normal', 'VIP', 'Emergency', 'Connection Critical'], weights=[74, 7, 3, 16])[0]

        for role, base_required in TASK_REQUIREMENTS.items():
            needed = max(1, int(np.ceil(base_required * AIRCRAFT_SIZE_FACTOR[aircraft_size])))
            task_start = arrival + timedelta(minutes=random.randint(-10, 35))
            duration = random.randint(18, 55)
            flights.append({
                'task_id': f'TASK-{i:03d}-{role[:3].upper()}',
                'flight_id': f'{random.choice(["AI", "6E", "SG", "UK", "QP"])}{random.randint(100, 999)}',
                'gate': gate,
                'aircraft_size': aircraft_size,
                'role': role,
                'required_staff': needed,
                'task_start': task_start,
                'task_end': task_start + timedelta(minutes=duration),
                'duration_min': duration,
                'priority': priority,
            })

    return pd.DataFrame(flights)


tasks_df = generate_flight_tasks()
tasks_df.head(12)

## 5. Fatigue and Shift Compliance

In [ ]:
def classify_fatigue(score):
    if score >= 82:
        return 'High'
    if score >= 60:
        return 'Medium'
    return 'Low'


def compliance_flags(row):
    flags = []
    if row['hours_worked'] > row['max_hours']:
        flags.append('daily_hours_exceeded')
    if row['continuous_hours'] > row['max_continuous_hours']:
        flags.append('break_required')
    if row['last_rest_hours'] < row['required_rest_hours']:
        flags.append('rest_non_compliant')
    if row['fatigue_score'] >= 82:
        flags.append('high_fatigue')
    return ','.join(flags) if flags else 'compliant'


staff_df['fatigue_level'] = staff_df['fatigue_score'].map(classify_fatigue)
staff_df['compliance_flags'] = staff_df.apply(compliance_flags, axis=1)
staff_df['eligible_for_assignment'] = (
    staff_df['status'].isin(['active', 'available']) &
    (staff_df['fatigue_score'] < 82) &
    (staff_df['hours_worked'] <= staff_df['max_hours']) &
    (staff_df['continuous_hours'] <= staff_df['max_continuous_hours'])
)

staff_df[['employee_id', 'name', 'role', 'status', 'hours_worked', 'continuous_hours', 'fatigue_score', 'fatigue_level', 'compliance_flags', 'eligible_for_assignment']].head(15)

## 6. Role Capacity and Shortage Detection

In [ ]:
def calculate_role_capacity(staff, tasks, horizon_minutes=90):
    horizon_end = now + timedelta(minutes=horizon_minutes)
    active_tasks = tasks[(tasks['task_start'] <= horizon_end) & (tasks['task_end'] >= now)].copy()
    demand = active_tasks.groupby('role')['required_staff'].sum().reindex(ROLES, fill_value=0)
    total = staff.groupby('role').size().reindex(ROLES, fill_value=0)
    eligible = staff[staff['eligible_for_assignment']].groupby('role').size().reindex(ROLES, fill_value=0)
    active = staff[staff['status'] == 'active'].groupby('role').size().reindex(ROLES, fill_value=0)
    high_fatigue = staff[staff['fatigue_level'] == 'High'].groupby('role').size().reindex(ROLES, fill_value=0)

    table = pd.DataFrame({
        'role': ROLES,
        'total_staff': total.values,
        'active_staff': active.values,
        'eligible_staff': eligible.values,
        'high_fatigue_staff': high_fatigue.values,
        'demand_next_90m': demand.values,
    })
    table['capacity_gap'] = table['eligible_staff'] - table['demand_next_90m']
    table['utilization_pct'] = np.where(table['eligible_staff'] > 0, table['demand_next_90m'] / table['eligible_staff'] * 100, 999)
    table['capacity_status'] = np.where(table['capacity_gap'] < 0, 'shortage', np.where(table['utilization_pct'] >= 85, 'tight', 'ok'))
    table['utilization_pct'] = table['utilization_pct'].round(1)
    return table.sort_values(['capacity_status', 'utilization_pct'], ascending=[False, False]).reset_index(drop=True)


capacity_df = calculate_role_capacity(staff_df, tasks_df)
capacity_df

## 7. Auto-Assignment Engine

In [ ]:
def assignment_score(employee, task):
    fatigue_penalty = employee['fatigue_score'] * 0.8
    hours_penalty = employee['hours_worked'] * 5
    gate_bonus = -8 if employee['current_gate'] == task['gate'] else 0
    standby_bonus = -5 if employee['current_gate'] == 'Standby' else 0
    priority_bonus = {'Normal': 0, 'VIP': -4, 'Emergency': -12, 'Connection Critical': -8}.get(task['priority'], 0)
    return fatigue_penalty + hours_penalty + gate_bonus + standby_bonus + priority_bonus


def auto_assign_crews(staff, tasks, max_tasks=80):
    available = staff[staff['eligible_for_assignment']].copy()
    assigned_employee_ids = set()
    assignments = []
    selected_tasks = tasks.sort_values(['task_start', 'priority']).head(max_tasks)

    for _, task in selected_tasks.iterrows():
        candidates = available[(available['role'] == task['role']) & (~available['employee_id'].isin(assigned_employee_ids))].copy()
        if candidates.empty:
            assignments.append({
                'task_id': task['task_id'],
                'flight_id': task['flight_id'],
                'gate': task['gate'],
                'role': task['role'],
                'required_staff': task['required_staff'],
                'assigned_count': 0,
                'assigned_employees': '',
                'assignment_status': 'unfilled',
            })
            continue

        candidates['score'] = candidates.apply(lambda row: assignment_score(row, task), axis=1)
        chosen = candidates.sort_values('score').head(int(task['required_staff']))
        assigned_employee_ids.update(chosen['employee_id'].tolist())
        status = 'filled' if len(chosen) >= task['required_staff'] else 'partial'

        assignments.append({
            'task_id': task['task_id'],
            'flight_id': task['flight_id'],
            'gate': task['gate'],
            'role': task['role'],
            'required_staff': task['required_staff'],
            'assigned_count': len(chosen),
            'assigned_employees': ','.join(chosen['employee_id'].tolist()),
            'assignment_status': status,
        })

    return pd.DataFrame(assignments)


assignments_df = auto_assign_crews(staff_df, tasks_df)
assignments_df.head(20)

## 8. HR Alerts and Recommendations

In [ ]:
def build_hr_recommendations(staff, capacity, assignments):
    alerts = []

    for _, row in capacity[capacity['capacity_status'] != 'ok'].iterrows():
        priority = 1 if row['capacity_status'] == 'shortage' else 2
        alerts.append({
            'priority_rank': priority,
            'type': 'role_capacity',
            'role': row['role'],
            'severity': 'High' if row['capacity_status'] == 'shortage' else 'Medium',
            'message': f"{row['role']} capacity is {row['capacity_status']}: demand {row['demand_next_90m']}, eligible {row['eligible_staff']}.",
            'recommendation': 'Call reserve crew or rebalance from low-demand terminal.' if row['capacity_status'] == 'shortage' else 'Keep standby crew near active gates.',
        })

    high_fatigue = staff[(staff['fatigue_level'] == 'High') & (staff['status'].isin(['active', 'available']))].sort_values('fatigue_score', ascending=False).head(12)
    for _, member in high_fatigue.iterrows():
        alerts.append({
            'priority_rank': 1,
            'type': 'fatigue_risk',
            'role': member['role'],
            'severity': 'High',
            'message': f"{member['employee_id']} {member['name']} has fatigue score {member['fatigue_score']}.",
            'recommendation': 'Move to break, replace on active assignment, and prevent new allocation until rested.',
        })

    non_compliant = staff[staff['compliance_flags'] != 'compliant'].head(12)
    for _, member in non_compliant.iterrows():
        alerts.append({
            'priority_rank': 2,
            'type': 'shift_compliance',
            'role': member['role'],
            'severity': 'Medium',
            'message': f"{member['employee_id']} flags: {member['compliance_flags']}.",
            'recommendation': 'Review roster, schedule rest, and avoid overtime extension.',
        })

    unfilled = assignments[assignments['assignment_status'].isin(['partial', 'unfilled'])].head(12)
    for _, task in unfilled.iterrows():
        alerts.append({
            'priority_rank': 1 if task['assignment_status'] == 'unfilled' else 2,
            'type': 'assignment_gap',
            'role': task['role'],
            'severity': 'High' if task['assignment_status'] == 'unfilled' else 'Medium',
            'message': f"{task['task_id']} for {task['flight_id']} at {task['gate']} is {task['assignment_status']}.",
            'recommendation': 'Escalate to operations desk and assign reserve crew immediately.',
        })

    return pd.DataFrame(alerts).sort_values(['priority_rank', 'severity']).reset_index(drop=True)


recommendations_df = build_hr_recommendations(staff_df, capacity_df, assignments_df)
recommendations_df.head(20)

## 9. Live Crew Activity Feed

In [ ]:
ACTIVITY_TYPES = ['Checked In', 'Assigned to Gate', 'Break', 'Shift End', 'Task Complete', 'Alert']


def generate_activity_feed(staff, assignments, n=30):
    feed = []
    staff_lookup = staff.set_index('employee_id')
    assigned_ids = []
    for employees in assignments['assigned_employees'].dropna():
        assigned_ids.extend([emp for emp in employees.split(',') if emp])
    candidate_ids = assigned_ids or staff['employee_id'].sample(min(n, len(staff)), random_state=SEED).tolist()

    for i in range(n):
        employee_id = random.choice(candidate_ids)
        if employee_id not in staff_lookup.index:
            continue
        member = staff_lookup.loc[employee_id]
        event_time = now - timedelta(minutes=random.randint(0, 90))
        activity = random.choice(ACTIVITY_TYPES)
        feed.append({
            'event_id': f'EVT-{1000 + i}',
            'time': event_time.strftime('%H:%M:%S'),
            'employee_id': employee_id,
            'name': member['name'],
            'role': member['role'],
            'activity': activity,
            'severity': 'High' if activity == 'Alert' else 'Normal',
        })

    return pd.DataFrame(feed).sort_values('time', ascending=False).reset_index(drop=True)


activity_df = generate_activity_feed(staff_df, assignments_df)
activity_df.head(15)

## 10. HR KPI Snapshot

In [ ]:
def build_hr_kpis(staff, capacity, assignments, recommendations):
    total_staff = len(staff)
    active_staff = int((staff['status'] == 'active').sum())
    available_staff = int((staff['status'] == 'available').sum())
    on_leave = int((staff['status'] == 'on_leave').sum())
    sick = int((staff['status'] == 'sick').sum())
    high_fatigue = int((staff['fatigue_level'] == 'High').sum())
    compliance_rate = (staff['compliance_flags'] == 'compliant').mean() * 100
    filled_rate = (assignments['assignment_status'] == 'filled').mean() * 100 if len(assignments) else 0
    shortage_roles = int((capacity['capacity_status'] == 'shortage').sum())
    avg_fatigue = staff['fatigue_score'].mean()
    hr_score = 100 - min(avg_fatigue * 0.35, 30) - min(high_fatigue * 0.7, 25) - min(shortage_roles * 6, 24) - min((100 - compliance_rate) * 0.25, 16)

    return pd.DataFrame([{
        'total_staff': total_staff,
        'active_staff': active_staff,
        'available_staff': available_staff,
        'on_leave': on_leave,
        'sick': sick,
        'high_fatigue_staff': high_fatigue,
        'avg_fatigue_score': round(avg_fatigue, 1),
        'shift_compliance_pct': round(compliance_rate, 1),
        'assignment_fill_rate_pct': round(filled_rate, 1),
        'shortage_roles': shortage_roles,
        'critical_alerts': int((recommendations['priority_rank'] == 1).sum()) if len(recommendations) else 0,
        'hr_operation_score': round(float(np.clip(hr_score, 0, 100)), 1),
    }])


kpi_df = build_hr_kpis(staff_df, capacity_df, assignments_df, recommendations_df)
kpi_df

## 11. HR Visuals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

role_counts = staff_df['role'].value_counts().reindex(ROLES, fill_value=0)
axes[0, 0].barh(role_counts.index, role_counts.values, color='#1565c0')
axes[0, 0].set_title('Staff by Role')
axes[0, 0].set_xlabel('Count')

fatigue_counts = staff_df['fatigue_level'].value_counts().reindex(['Low', 'Medium', 'High'], fill_value=0)
axes[0, 1].bar(fatigue_counts.index, fatigue_counts.values, color=['#2e7d32', '#f9a825', '#c62828'])
axes[0, 1].set_title('Fatigue Levels')

axes[1, 0].barh(capacity_df['role'], capacity_df['utilization_pct'], color='#00897b')
axes[1, 0].axvline(85, color='#f9a825', linestyle='--', label='Tight')
axes[1, 0].axvline(100, color='#c62828', linestyle='--', label='Shortage')
axes[1, 0].set_title('Role Utilization Pressure')
axes[1, 0].set_xlabel('Utilization %')
axes[1, 0].legend()

assignment_counts = assignments_df['assignment_status'].value_counts().reindex(['filled', 'partial', 'unfilled'], fill_value=0)
axes[1, 1].bar(assignment_counts.index, assignment_counts.values, color=['#2e7d32', '#f9a825', '#c62828'])
axes[1, 1].set_title('Assignment Status')

plt.tight_layout()
plt.show()

## 12. Backend-Ready Payload

In [ ]:
def build_hr_payload(staff, capacity, assignments, recommendations, activity, kpis):
    roster_columns = [
        'employee_id', 'name', 'role', 'status', 'shift', 'hours_worked', 'continuous_hours',
        'fatigue_score', 'fatigue_level', 'compliance_flags', 'eligible_for_assignment', 'current_gate'
    ]
    return {
        'generated_at': now.isoformat(),
        'kpis': kpis.iloc[0].to_dict(),
        'staff_roster': staff[roster_columns].to_dict(orient='records'),
        'role_capacity': capacity.to_dict(orient='records'),
        'assignments': assignments.head(50).to_dict(orient='records'),
        'recommendations': recommendations.head(20).to_dict(orient='records'),
        'activity_feed': activity.head(20).to_dict(orient='records'),
        'fatigue_watchlist': staff.sort_values('fatigue_score', ascending=False).head(15)[roster_columns].to_dict(orient='records'),
    }


payload = build_hr_payload(staff_df, capacity_df, assignments_df, recommendations_df, activity_df, kpi_df)
payload

In [ ]:
print('HUMAN RESOURCES SUMMARY')
print('=======================')
print(f"Total staff: {payload['kpis']['total_staff']}")
print(f"Active staff: {payload['kpis']['active_staff']}")
print(f"High fatigue staff: {payload['kpis']['high_fatigue_staff']}")
print(f"Shift compliance: {payload['kpis']['shift_compliance_pct']}%")
print(f"Assignment fill rate: {payload['kpis']['assignment_fill_rate_pct']}%")
print(f"Shortage roles: {payload['kpis']['shortage_roles']}")
print(f"HR Operation Score: {payload['kpis']['hr_operation_score']}")
print('\nTop HR recommendations:')
for item in payload['recommendations'][:5]:
    print(f"- [{item['type']}] {item['message']} {item['recommendation']}")